# Loading the data

In [45]:
import os
import torch
# __file__ isn't defined in notebooks, so resolve relative to the notebook's cwd
BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, 'DATA', 'input.txt')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Loaded {len(text)} characters from {DATA_PATH}")

Loaded 1115394 characters from /Users/I578070/Desktop/NANOGPT/DATA/input.txt


In [46]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [47]:
char = sorted(set(text))
vocab_size = len(char)

print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 65


# Encoder and Decoder

In [48]:
stou = {ch: i for i, ch in enumerate(char)}
utos = {i: ch for i, ch in enumerate(char)}
encode = lambda s: [stou[ch] for ch in s]
decode = lambda l: ''.join([utos[i] for i in l])

print(encode("Hii there"))
print(decode(encode("Hii there")))

[20, 47, 47, 1, 58, 46, 43, 56, 43]
Hii there


# coverting the data into tensor

In [49]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape,data.dtype)

torch.Size([1115394]) torch.int64


# Spliting the data for train and test

In [50]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

print(train_data.shape, train_data.dtype)
print(val_data.shape, val_data.dtype)

torch.Size([1003854]) torch.int64
torch.Size([111540]) torch.int64


In [51]:
block_size = 64
x = train_data[:block_size]
y = val_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target = y[i]
    print(f"when in iteration {i}, context: {context}, target: {target}")

when in iteration 0, context: tensor([18]), target: 0
when in iteration 1, context: tensor([18, 47]), target: 0
when in iteration 2, context: tensor([18, 47, 56]), target: 19
when in iteration 3, context: tensor([18, 47, 56, 57]), target: 30
when in iteration 4, context: tensor([18, 47, 56, 57, 58]), target: 17
when in iteration 5, context: tensor([18, 47, 56, 57, 58,  1]), target: 25
when in iteration 6, context: tensor([18, 47, 56, 57, 58,  1, 15]), target: 21
when in iteration 7, context: tensor([18, 47, 56, 57, 58,  1, 15, 47]), target: 27
when in iteration 8, context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58]), target: 10
when in iteration 9, context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47]), target: 0
when in iteration 10, context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64]), target: 19
when in iteration 11, context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43]), target: 53
when in iteration 12, context: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47

In [52]:
torch.manual_seed(42)
batch_size = 16
block_size = 32

def get_batch(split):
    #generate a small batch of data of input and target sequences
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch('train')       
print(x.shape, y.shape)

for b in range(batch_size):
    for t in range(block_size):
        context = x[b, :t+1]
        target = y[b, t]
        print(f"when in batch {b}, time step {t}, context: {context}, target: {target}")

torch.Size([16, 32]) torch.Size([16, 32])
when in batch 0, time step 0, context: tensor([46]), target: 43
when in batch 0, time step 1, context: tensor([46, 43]), target: 5
when in batch 0, time step 2, context: tensor([46, 43,  5]), target: 57
when in batch 0, time step 3, context: tensor([46, 43,  5, 57]), target: 1
when in batch 0, time step 4, context: tensor([46, 43,  5, 57,  1]), target: 42
when in batch 0, time step 5, context: tensor([46, 43,  5, 57,  1, 42]), target: 43
when in batch 0, time step 6, context: tensor([46, 43,  5, 57,  1, 42, 43]), target: 39
when in batch 0, time step 7, context: tensor([46, 43,  5, 57,  1, 42, 43, 39]), target: 42
when in batch 0, time step 8, context: tensor([46, 43,  5, 57,  1, 42, 43, 39, 42]), target: 6
when in batch 0, time step 9, context: tensor([46, 43,  5, 57,  1, 42, 43, 39, 42,  6]), target: 1
when in batch 0, time step 10, context: tensor([46, 43,  5, 57,  1, 42, 43, 39, 42,  6,  1]), target: 46
when in batch 0, time step 11, contex

# BigRamLanguageModel

In [53]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx,target=None):
        logits = self.token_embedding_table(idx)
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #idx is the current sequence of token indices
        for _ in range(max_new_tokens):
            logits,loss = self(idx)
            B,T,C = logits.shape
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)
logits, loss = m.forward(torch.tensor([[0]]).to(device), None)
print(loss)

print(decode(m.generate(torch.tensor([[0]]).to(device), max_new_tokens=10)[0].tolist()))


None

o$,q&IWqW&


In [54]:
@torch.no_grad()
def estimate_loss(model, eval_iters=100):
    model.eval()
    losses = {}
    for split in ('train', 'val'):
        split_losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x.to(device), y.to(device))
            split_losses[k] = loss.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses


def train(model, steps=1000, learning_rate=1e-3, eval_interval=100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()

    for step in range(steps):
        x, y = get_batch('train')
        x, y = x.to(device), y.to(device)

        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % eval_interval == 0 or step == steps - 1:
            losses = estimate_loss(model)
            print(
                f"step {step}: "
                f"train loss {losses['train']:.4f}, "
                f"val loss {losses['val']:.4f}"
            )

    return model


m = train(m, steps=1000)


step 0: train loss 4.7534, val loss 4.7627
step 100: train loss 4.6313, val loss 4.6322
step 200: train loss 4.5111, val loss 4.5201
step 300: train loss 4.3834, val loss 4.3995
step 400: train loss 4.2802, val loss 4.2888
step 500: train loss 4.1756, val loss 4.1808
step 600: train loss 4.0731, val loss 4.0830
step 700: train loss 3.9740, val loss 3.9882
step 800: train loss 3.8793, val loss 3.8990
step 900: train loss 3.7900, val loss 3.8117
step 999: train loss 3.7155, val loss 3.7237


In [56]:
prompt = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_tokens = m.generate(prompt, max_new_tokens=100)
print(decode(generated_tokens[0].tolist()))


imseM
W
CS?;gefdl;flfdTaru mWw'gZMffHMpNcV!Zn'dVIt&aSVzOMBri?mmLI;ffFCO,gemgNw'XtjvSNk wPeX.FMuetGmW
